# Notebook 01 · Data: CQ500, BHX and CT-ICH on Google Drive

This notebook:

1. downloads the **BHX boxes** (PhysioNet) and the **CQ500 images** (qure.ai, 28.7 GB) to **private**
   Google Drive folders, one per dataset:

   ```
   MyDrive/ich-benchmark/dados/
   └── tc/
       ├── cq500/   zips/, zips_recuperados/, reads.csv, zips_corrompidos.json, cq500_mapa_exames.csv
       └── bhx/     boxes (CSV files) and bhx_x_cq500.json
   ```

2. checks the BHX checksums and the size of every CQ500 archive;
3. **tests every archive** and repairs those that are corrupted at the source (CQ500-CT-120, found on
   17 Oct 2026);
4. **matches the examinations**: reads only the header of one DICOM file per examination (without
   unzipping) to link each CQ500 name to the BHX `StudyInstanceUID`;
5. compares the **226 examinations with BHX boxes** with the reads of the three CQ500 radiologists;
6. downloads and describes the CT-ICH test set (restricted access, see below).

**Licences:** CQ500, CC BY-NC-SA 4.0 and the qure.ai end-user licence (non-commercial use only; derived
annotations under the same licence). BHX, CC BY-SA 4.0. By running this notebook you accept these licences.

**Environment:** a CPU runtime is enough. The notebook can be run again safely: whatever was already
downloaded and checked is skipped. Network failures are expected; the cells retry by themselves.

**Language:** documentation and comments are in English; identifiers, printed messages and the column
names of the files written here are in Portuguese, exactly as in the run that produced the results.

In [ ]:
# 1. Mount the Drive and define the (private) data folders: one folder per dataset
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DADOS = Path('/content/drive/MyDrive/ich-benchmark/dados')
BHX, CQ500 = DADOS / 'tc' / 'bhx', DADOS / 'tc' / 'cq500'
for p in (BHX, CQ500 / 'zips', CQ500 / 'zips_recuperados'):
    p.mkdir(parents=True, exist_ok=True)
LOCAL = Path('/content/tmp')                           # Colab disk: fast, erased at the end of the session
LOCAL.mkdir(exist_ok=True)
print(BHX, CQ500, sep='\n')

In [ ]:
# 2. Download functions (used by the following cells)
#    Downloads to the local disk and only then moves to the Drive; up to 5 attempts per file.
import hashlib, shutil, time, urllib.request

def com_tentativas(funcao, *args, tentativas=5):
    for n in range(1, tentativas + 1):
        try:
            return funcao(*args)
        except Exception:
            if n == tentativas:
                raise
            time.sleep(5 * n)                          # waits 5, 10, 15, 20 s

def _copiar(url, destino, tempo_limite):
    tmp = LOCAL / destino.name
    with urllib.request.urlopen(url, timeout=tempo_limite) as r, open(tmp, 'wb') as f:
        shutil.copyfileobj(r, f, 1 << 20)
    shutil.move(tmp, destino)

def baixar(url, destino, tempo_limite=300):
    com_tentativas(_copiar, url, destino, tempo_limite)

def tamanho_remoto(url):
    def _head():
        with urllib.request.urlopen(urllib.request.Request(url, method='HEAD'), timeout=60) as r:
            return int(r.headers['Content-Length'])
    return com_tentativas(_head)

In [ ]:
# 3. BHX v1.1 boxes (PhysioNet) and check against the published checksums
#    PhysioNet is slow: this may take up to ~10 minutes. A file is skipped only if its checksum matches.
URL_BHX = 'https://physionet.org/files/bhx-brain-bounding-box/1.1/'
CSVS = ['1_Initial_Manual_Labeling.csv', '2_Extrapolation_to_All_Series.csv', '3_Extrapolation_to_Selected_Series.csv']

for nome in ('SHA256SUMS.txt', 'LICENSE.txt'):
    baixar(URL_BHX + nome, BHX / nome)
esperado = {l.split()[1].lstrip('*'): l.split()[0] for l in (BHX / 'SHA256SUMS.txt').read_text().splitlines() if l.strip()}
confere = lambda nome: (BHX / nome).exists() and hashlib.sha256((BHX / nome).read_bytes()).hexdigest() == esperado[nome]

for nome in CSVS:
    if not confere(nome):
        print('baixando', nome, '...', flush=True)
        baixar(URL_BHX + nome, BHX / nome)
    print(('OK  ' if confere(nome) else 'ERRO'), nome)

In [ ]:
# 4. CQ500 images (491 archives, 28.7 GB) and radiologists' reads
#    Runs in parallel, resumes where it stopped and checks sizes. One error does not stop the other files.
#    If any ERRO remains at the end, run this cell again.
import collections, concurrent.futures as cf

URL_CQ = 'https://s3.ap-south-1.amazonaws.com/qure.headct.study/'
ZIPS = [f'CQ500-CT-{i}.zip' for i in range(491)]

if not (CQ500 / 'reads.csv').exists():
    baixar(URL_CQ + 'reads.csv', CQ500 / 'reads.csv')

def baixar_zip(nome):
    destino = CQ500 / 'zips' / nome
    try:
        total = tamanho_remoto(URL_CQ + nome)
        if destino.exists() and destino.stat().st_size == total:
            return nome, 'já existia'
        baixar(URL_CQ + nome, destino, tempo_limite=120)
        return nome, 'baixado' if destino.stat().st_size == total else 'ERRO de tamanho'
    except Exception as erro:
        return nome, f'ERRO: {type(erro).__name__}'

resultados, inicio = [], time.time()
with cf.ThreadPoolExecutor(4) as ex:
    for k, (nome, status) in enumerate(ex.map(baixar_zip, ZIPS), 1):
        resultados.append((nome, status))
        if k % 25 == 0 or status.startswith('ERRO'):
            print(f'{k}/491  {time.time() - inicio:5.0f} s  último: {nome} {status}', flush=True)

print(collections.Counter(s.split(':')[0] for _, s in resultados))
print('Com erro (rode a célula de novo):', [n for n, s in resultados if s.startswith('ERRO')])

In [ ]:
# 5. Test every archive and repair the corrupted ones
#    A correct size does not guarantee the content: CQ500-CT-120.zip comes from the server without its
#    central directory (found on 17 Oct 2026). For each archive that does not open: (a) download again outside the Drive; (b) if
#    it is still broken, the defect is at the source and `zip -FF` recovers the intact DICOM files into a copy in
#    zips_recuperados/. The original is left untouched. Everything is recorded in zips_corrompidos.json.
import json, subprocess, zipfile

def abre(caminho):
    try:
        with zipfile.ZipFile(caminho) as z:
            return sum(m.lower().endswith('.dcm') for m in z.namelist())
    except zipfile.BadZipFile:
        return None

ruins = [n for n in ZIPS if abre(CQ500 / 'zips' / n) is None]
print(len(ruins), 'zip(s) que não abrem:', ruins)

registro = {}
for nome in ruins:
    recuperado = CQ500 / 'zips_recuperados' / nome
    if recuperado.exists() and abre(recuperado):
        registro[nome] = {'situacao': 'recuperado antes', 'dicoms_recuperados': abre(recuperado)}
        continue
    novo = LOCAL / nome
    baixar(URL_CQ + nome, novo)
    if abre(novo) is not None:
        shutil.copy(novo, CQ500 / 'zips' / nome)
        registro[nome] = {'situacao': 'estragou no download; baixado de novo', 'dicoms': abre(novo)}
        continue
    # zip -FF asks "Is this a single-disk archive? (y/n)": the answer is passed as input
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'zip'], check=True, capture_output=True)
    corrigido = LOCAL / nome.replace('.zip', '-recuperado.zip')
    subprocess.run(['zip', '-FF', str(novo), '--out', str(corrigido)], input=b'y\n', capture_output=True)
    # Even after repair, the last file may be truncated (in CT-120, CT000168.dcm):
    # build a copy with only the files that unzip without error.
    limpo, descartados = LOCAL / nome.replace('.zip', '-limpo.zip'), []
    if abre(corrigido) is not None:
        with zipfile.ZipFile(corrigido) as z, zipfile.ZipFile(limpo, 'w', zipfile.ZIP_DEFLATED) as saida:
            for m in z.namelist():
                try:
                    saida.writestr(m, z.read(m))
                except Exception:
                    descartados.append(m)
    n = abre(limpo) if limpo.exists() else 0
    if n:
        shutil.copy(limpo, recuperado)
    registro[nome] = {'situacao': 'corrompido no servidor; recuperado com zip -FF' if n else 'corrompido no servidor; sem recuperação',
                      'dicoms_recuperados': n or 0, 'arquivos_descartados': descartados}

(CQ500 / 'zips_corrompidos.json').write_text(json.dumps(registro, indent=2, ensure_ascii=False))
print(json.dumps(registro, indent=2, ensure_ascii=False))

In [ ]:
# 6. Link each CQ500 examination to its StudyInstanceUID (reads 1 DICOM header per archive, without unzipping)
#    Uses the repaired copy when the original is corrupted. A broken archive becomes a row with 'erro' (error).
import csv, io, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pydicom'], check=True)
import pydicom

def zip_do_exame(nome_zip):
    recuperado = CQ500 / 'zips_recuperados' / nome_zip
    return (recuperado, 'sim') if recuperado.exists() else (CQ500 / 'zips' / nome_zip, '')

def identificar(i):
    nome = f'CQ500-CT-{i}'
    caminho, usou_recuperado = zip_do_exame(f'{nome}.zip')
    try:
        with zipfile.ZipFile(caminho) as z:
            dicoms = [m for m in z.namelist() if m.lower().endswith('.dcm')]
            cab = pydicom.dcmread(io.BytesIO(z.read(dicoms[0])), stop_before_pixels=True)
            series = {m.rsplit('/', 2)[-2] for m in dicoms}
        return {'Name': nome, 'StudyInstanceUID': cab.StudyInstanceUID, 'dicoms': len(dicoms),
                'pastas_de_serie': len(series), 'zip_recuperado': usou_recuperado, 'erro': ''}
    except Exception as erro:
        return {'Name': nome, 'StudyInstanceUID': '', 'dicoms': 0, 'pastas_de_serie': 0,
                'zip_recuperado': usou_recuperado, 'erro': f'{type(erro).__name__}: {erro}'}

with cf.ThreadPoolExecutor(8) as ex:
    mapa = list(ex.map(identificar, range(491)))
with open(CQ500 / 'cq500_mapa_exames.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=mapa[0].keys()); w.writeheader(); w.writerows(mapa)
print(len(mapa), 'exames;', sum(m['dicoms'] for m in mapa), 'arquivos DICOM')
print('Com erro:', [(m['Name'], m['erro']) for m in mapa if m['erro']])

In [ ]:
# 7. BHX versus CQ500 reads: how many radiologists reported haemorrhage in the 226 examinations with boxes?
leituras = {l['name']: l for l in csv.DictReader(open(CQ500 / 'reads.csv'))}   # the column is 'name', in lower case
uid_para_nome = {m['StudyInstanceUID']: m['Name'] for m in mapa if m['StudyInstanceUID']}
bhx = list(csv.DictReader(open(BHX / '3_Extrapolation_to_Selected_Series.csv')))
estudos_bhx = {l['StudyInstanceUID'] for l in bhx}

votos = lambda nome: sum(int(leituras[nome][f'R{r}:ICH']) for r in (1, 2, 3))
sem_par = sorted(u for u in estudos_bhx if u not in uid_para_nome)
com_caixa = {uid_para_nome[u] for u in estudos_bhx if u in uid_para_nome}
resumo = {
    'exames_bhx': len(estudos_bhx),
    'exames_bhx_sem_par_no_cq500': len(sem_par),
    'votos_ich_nos_exames_com_caixa': dict(collections.Counter(votos(n) for n in com_caixa)),
    'exames_maioria_ich_sem_caixa': sorted(n for n in leituras if votos(n) >= 2 and n not in com_caixa),
    'exames_cq500': len(leituras),
    'zips_com_erro': [m['Name'] for m in mapa if m['erro']],
    'zips_recuperados': [m['Name'] for m in mapa if m['zip_recuperado']],
}
(BHX / 'bhx_x_cq500.json').write_text(json.dumps(resumo, indent=2, ensure_ascii=False))
print(json.dumps({k: v if not isinstance(v, list) else f'{len(v)} exames: {v[:10]}' for k, v in resumo.items()}, indent=2, ensure_ascii=False))

## CT-ICH (test set): download and description

Dataset from a hospital in Iraq (Hssayeni, 2020; PhysioNet, doi:10.13026/4nae-zg36): 75 examinations in
NIfTI format, with the haemorrhage outlined on each slice. **Restricted access:** only users with a
PhysioNet account who signed the PhysioNet Restricted Health Data Use Agreement 1.5.0. Under this
agreement **the data cannot be shared** with anyone who has not signed it.

The password is requested at run time and is not stored in the notebook or on the Drive.

In [ ]:
# 8. Download CT-ICH v1.3.1 from PhysioNet (login required), in parallel, directly to the Drive
#    Requires cells 1 and 2. Downloads 3 files at a time with wget (PhysioNet
#    rejects the Python standard library with error 403); as soon as a file arrives, its published checksum
#    is checked and the file is copied to the Drive. If the session drops, a new run downloads only what is missing.
#    Files already downloaded to /content/tmp/ct-ich in an earlier run are reused.
import collections, csv, getpass, hashlib, json, shutil, subprocess, sys, time
import concurrent.futures as cf
CTICH = DADOS / 'tc' / 'ct-ich'
CTICH.mkdir(parents=True, exist_ok=True)
URL_CTICH = 'https://physionet.org/files/ct-ich/1.3.1/'
PARTES = LOCAL / 'ctich_partes'               # downloads in progress (Colab disk)
ANTIGO = LOCAL / 'ct-ich'                     # what an earlier run already downloaded here
PARTES.mkdir(parents=True, exist_ok=True)

usuario = input('Usuário do PhysioNet (o nome de usuário, não o e-mail): ')
senha = getpass.getpass('Senha do PhysioNet (não fica gravada): ')

def sha256(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 20), b''):
            h.update(bloco)
    return h.hexdigest()

def baixar_autenticado(nome, destino):
    def _baixar():
        r = subprocess.run(['wget', '-q', '-O', str(destino), '--user', usuario, '--password', senha,
                            URL_CTICH + nome], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f'wget código {r.returncode}')
    com_tentativas(_baixar)

try:
    # 1) the official list of files and checksums
    baixar_autenticado('SHA256SUMS.txt', CTICH / 'SHA256SUMS.txt')
    esperado = {}
    for l in (CTICH / 'SHA256SUMS.txt').read_text().splitlines():
        if l.strip():
            soma, nome = l.split(maxsplit=1)
            esperado[nome.strip().lstrip('*')] = soma
    print(len(esperado), 'arquivos na lista oficial')

    # 2) one file: already on the Drive? downloaded before? otherwise download, check and copy to the Drive
    def obter(nome):
        destino = CTICH / nome
        try:
            if destino.exists() and sha256(destino) == esperado[nome]:
                return nome, 'já no Drive'
            destino.parent.mkdir(parents=True, exist_ok=True)
            anterior = ANTIGO / nome
            if anterior.exists() and sha256(anterior) == esperado[nome]:
                shutil.copy(anterior, destino)
                return nome, 'reaproveitado'
            parte = PARTES / nome.replace('/', '__')
            baixar_autenticado(nome, parte)
            if sha256(parte) != esperado[nome]:
                parte.unlink(); return nome, 'ERRO de checksum'
            shutil.move(parte, destino)
            return nome, 'baixado'
        except Exception as erro:
            return nome, f'ERRO: {type(erro).__name__} {erro}'[:120]

    resultados, inicio = [], time.time()
    # 3 connections: with more, PhysioNet returned errors (wget code 8, 18 Oct 2026)
    with cf.ThreadPoolExecutor(3) as ex:
        for k, (nome, status) in enumerate(ex.map(obter, sorted(esperado)), 1):
            resultados.append((nome, status))
            if status in ('baixado', 'reaproveitado') or status.startswith('ERRO') or k % 20 == 0:
                print(f'{k}/{len(esperado)}  {time.time() - inicio:5.0f} s  {nome}: {status}', flush=True)
finally:
    del senha                                  # the password is removed from memory even if an error occurs

print(collections.Counter(s.split(':')[0] for _, s in resultados))
falta = [n for n, s in resultados if s.startswith('ERRO')]
print('Tudo certo em ' + str(CTICH) if not falta else f'{len(falta)} com erro — rode a célula de novo: {falta[:10]}')

In [ ]:
# 9. Describe CT-ICH: format, spacing, whether intensities are in HU and what the masks look like
#    Reads the volumes from the Drive in parallel (8 at a time).
import collections, csv, json, statistics, subprocess, sys
import concurrent.futures as cf
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'nibabel'], check=True)
import nibabel as nib, numpy as np

volumes = sorted(p for p in CTICH.rglob('*.nii*') if 'mask' not in str(p).lower())
mascaras = {p.name: p for p in CTICH.rglob('*.nii*') if 'mask' in str(p).lower()}
print(len(volumes), 'volumes de TC;', len(mascaras), 'máscaras')

def descrever(v):
    img = nib.load(v)
    dados_v = img.get_fdata(dtype=np.float32)
    m = mascaras.get(v.name)
    dados_m = nib.load(m).get_fdata(dtype=np.float32) if m else None
    return {
        'exame': v.name, 'forma': list(img.shape),
        'espacamento_mm': [round(float(x), 3) for x in img.header.get_zooms()[:3]],
        'intensidade_min': float(dados_v.min()), 'intensidade_max': float(dados_v.max()),
        'tipo': str(img.get_data_dtype()),
        'mascara': bool(m),
        'valores_da_mascara': sorted({int(x) for x in np.unique(dados_m)}) if m else [],
        'cortes_com_sangramento': int((dados_m.reshape(-1, dados_m.shape[-1]) > 0).any(axis=0).sum()) if m else 0,
    }

with cf.ThreadPoolExecutor(8) as ex:
    linhas = list(ex.map(descrever, volumes))

mins, maxs = [l['intensidade_min'] for l in linhas], [l['intensidade_max'] for l in linhas]
resumo = {
    'volumes': len(linhas), 'com_mascara': sum(l['mascara'] for l in linhas),
    'com_sangramento_na_mascara': sum(l['cortes_com_sangramento'] > 0 for l in linhas),
    'intensidade_min_mediana': statistics.median(mins), 'intensidade_max_mediana': statistics.median(maxs),
    'parece_HU': statistics.median(mins) <= -500 and statistics.median(maxs) >= 1000,
    'valores_de_mascara_encontrados': sorted({v for l in linhas for v in l['valores_da_mascara']}),
    'espacamentos': sorted({tuple(l['espacamento_mm']) for l in linhas})[:10],
    'csvs': sorted(p.name for p in CTICH.rglob('*.csv')),
}
with open(CTICH / 'ctich_descricao_exames.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=linhas[0].keys()); w.writeheader(); w.writerows(linhas)
(CTICH / 'ctich_descricao.json').write_text(json.dumps(resumo, indent=2, ensure_ascii=False, default=list))
print(json.dumps(resumo, indent=2, ensure_ascii=False, default=list))

## After running

The small files written to the Drive (`tc/cq500/cq500_mapa_exames.csv`, `tc/cq500/zips_corrompidos.json`,
`tc/bhx/bhx_x_cq500.json`, `tc/ct-ich/ctich_descricao.json`) document the downloads and explain the
difference between the CQ500 examinations positive by majority read and the 226 examinations with BHX
boxes. The images stay **only on the Drive**.